The notebook shows an intriguing property of AI generated images. When projected in CLIP's embedding space, they are narurally separated along
the main geometrical components of the data manifold.

We compare AI-pastiche vs a set of paintings of the National Gallery of Washington, but similar results hold for different datasets.

## Download AI-Pastiche

AI-Pastiche is a public dataset on Kaggle. You need your kaggle.json registration.

In [1]:
from google.colab import files
import json
import pandas as pd
import os
import ipywidgets as widgets
# Import the Image class from the PIL library
from PIL import Image, ImageOps
import numpy as np
import torch
from torchvision import transforms as T
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', None)

AI-Pastiche is freely available on kaggle. You need a token, though.

In [3]:
uploaded = files.upload()

Saving kaggle.json to kaggle (1).json


In [4]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [5]:
ls ~/.config/kaggle

kaggle.json


In [6]:
from kaggle.api.kaggle_api_extended import KaggleApi

In [7]:
"Define metadata for the Kaggle dataset"

dataset_metadata = {
    #"title": "DeepFakeDB",  # Replace with your dataset title
    "title": "AI-pastiche",
    "id":"asperticsuniboit/DeepFakeDataBase",  # Replace with your Kaggle username and dataset name
    "licenses": [{"name": "CC0-1.0"}]  # License for your dataset
}

Downloading the dataset takes about half a minute.

In [8]:
import gdown
!gdown '1UMnZU_xcVM21-bU1dz1o34aloyHYcr37'
csv = "/content/metadata_with_summary.csv"
# Initialize the Kaggle API
api = KaggleApi()
api.authenticate()

# Function to check if a dataset exists on Kaggle
def dataset_exists_on_kaggle(dataset_id, dataset_title):
    try:
      api.dataset_metadata(dataset_id,dataset_title)
      print("Dataset exists on Kaggle")
      return True
    except:
      print("Dataset does not exist on Kaggle")
      return False

# Dataset info
dataset_id = dataset_metadata["id"]
dataset_title = dataset_metadata["title"]
csv_file_name = "metadata.csv"

dataset_exists = dataset_exists_on_kaggle(dataset_id, dataset_title)
if not dataset_exists:
  metadata = pd.DataFrame(columns=["generative_model", "subject", "style", "period", "prompt", "generated_image"])
  metadata.head()
else:
  api.dataset_download_files(dataset_id, path='./', unzip=True)
  # Read the metadata if it exists

AIpastiche = pd.read_csv(csv)

Downloading...
From: https://drive.google.com/uc?id=1UMnZU_xcVM21-bU1dz1o34aloyHYcr37
To: /content/metadata_with_summary.csv
100% 1.06M/1.06M [00:00<00:00, 147MB/s]
Dataset exists on Kaggle
Dataset URL: https://www.kaggle.com/datasets/asperticsuniboit/DeepFakeDataBase


## Importing NGAD

The subset of paintings of the National Gallery of Art of Washington (NGAD) that we use is available in the following folder on google drive: https://drive.google.com/drive/folders/1jODECLejzIWuA1ECvnOWeghyclzZOIB5?usp=sharing

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
drive_path = '/content/drive/MyDrive/NGAD_repository'
file_name = "utility.py"
! cp "{drive_path}"/"{file_name}" /content/"{file_name}"

import utility

import importlib
importlib.reload(utility)
###
!pip install git+https://github.com/openai/CLIP.git
import pandas as pd
import os
import clip
import torch
torch.manual_seed(42)

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-0tu1wt_8
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-0tu1wt_8
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=2b843ee6575c02927ce409a3ae2a55865cb6a9d50f29e50f356f984c77ca2a35
  Stored in directory: /tmp/pip-ephem-wheel-cache-7cgcoyfu/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [12]:
# Read the National Gallery dataset from CSV file
NGD = pd.read_csv(f'{drive_path}/national_gallery_dataset_expanded_with_style.csv')
NGD = NGD[NGD["media"] == "Painting"]
NGD = NGD.sample(len(AIpastiche), random_state=43)
image_indices = NGD.index.tolist()
#print(NGD)

# Computing CLIP embeddings

In [ ]:
from tqdm import tqdm  # Per barra di progresso
import requests
from io import BytesIO

def get_image_features(image_paths, batch_size=32):
    image_features = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing images"):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = []

        for p in batch_paths:
            img_pil = Image.open(p).convert("RGB")
            img = preprocess(img_pil)  # returns tensor
            batch_images.append(img)

        if batch_images: # Process batch only if there are valid images
            batch_tensor = torch.stack(batch_images).to(device)

            with torch.no_grad():
                batch_features = model.encode_image(batch_tensor)
                batch_features /= batch_features.norm(dim=-1, keepdim=True)  # Normalizza
                image_features.append(batch_features.cpu())  # Salviamo su CPU per evitare OOM

    return torch.cat(image_features)  # Concateniamo tutti gli embeddings


def get_text_features(texts, batch_size=32):
    text_features = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Processing texts"):
        batch_texts = clip.tokenize(texts[i:i + batch_size]).to(device)

        with torch.no_grad():
            batch_features = model.encode_text(batch_texts)
            batch_features /= batch_features.norm(dim=-1, keepdim=True)  # Normalizza
            text_features.append(batch_features.cpu())  # Salviamo su CPU

    return torch.cat(text_features)

AI-Pastiche embeddings

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

load_emb = True   #if you want to reuse precomputed embeddings
for m in ['ViT-L/14@336px']:
  model, preprocess = clip.load(m, device=device)
  #model = model.float()  # this converts all layers, including conv1, to float32

  image_path = f'./generated_images'

  image_list = []

  # Aggiungi immagini e testi alla lista
  for _, d in AIpastiche.iterrows():
      image_list.append(f"{image_path}/{d['generated_image']}")  # Salviamo solo il path per ora

  if not load_emb:
    print("Computing AI-Pastiche image features")
    image_features_AI = get_image_features(image_list, batch_size=16)
    print(f"AI-pastiche shape: {image_features_AI.shape}")
    np.save("image_features_AI_Pastiche.npy", image_features_AI.numpy())
    #utility.save_embeddings(m, drive_path, image_features_AI, "image_features_AI_Pastiche.pt")
  else:
    print("Loading AI-Pastiche image features")
    #image_features_AI = utility.load_embeddings(m, drive_path, "image_features_AI_Pastiche.pt")
    image_features_AI = np.load('image_features_AI_Pastiche.npy')
    print(f"AI-pastiche shape: {image_features_AI.shape}")


Loading AI-Pastiche image features
AI-pastiche shape: (953, 768)


NGDA embeddings

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

load_nga_emb = True
for m in ['ViT-L/14@336px']:
  model, preprocess = clip.load(m, device=device)
  #model = model.float()  # this converts all layers, including conv1, to float32

  nga_image_path = f'{drive_path}/national_gallery_images'

  nga_image_list = []

  for _, d in NGD.iterrows():
      nga_image_list.append(f"{nga_image_path}/{d['objectid']}.jpg")


  if load_nga_emb and os.path.exists(f"{drive_path}/embeddings/{m.replace('/', '-')}"):
    print(f"Loading NGAD embeddings for model {m}")
    # load embeddings
    #image_features_NGAD = utility.load_embeddings(m, drive_path, "image_features_style.pt")
    #image_features_NGAD = image_features_NGAD[image_indices]
    image_features_NGAD = np.load('image_features_NGAD.npy')
    print(f"NGAD shape: {image_features_NGAD.shape}")
  else:
    print("Computing NGAD image features")
    image_features_NGAD = get_image_features(nga_image_list, batch_size=16)
    print(f"NGAD shape: {image_features_NGAD.shape}")
    np.save("image_features_NGAD.npy", image_features_NGAD.numpy())

Loading NGAD embeddings for model ViT-L/14@336px
NGAD shape: (953, 768)


## Projection

In [ ]:
# combine image_features
image_features = torch.tensor(np.concatenate((image_features_NGAD, image_features_AI)))
print(image_features.shape)

torch.Size([1906, 768])


In [ ]:
def print_test_results(model, test_features, test_labels):
    result = model.predict(test_features)
    print("Accuracy:", accuracy_score(test_labels, result))
    print("Precision:", precision_score(test_labels, result, average='weighted', zero_division=0))
    print("Recall:", recall_score(test_labels, result, average='weighted'))
    print("F1:", f1_score(test_labels, result, average='weighted'))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

mixed_df = pd.concat([NGD, AIpastiche], ignore_index=True)

NGD = NGD.reset_index(drop=True)
AIpastiche = AIpastiche.reset_index(drop=True)

# Split AIpastiche
groups = AIpastiche["prompt"]
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_indices_ai, test_indices_ai = next(gss.split(AIpastiche, groups=groups))

df_train_aipastiche = AIpastiche.iloc[train_indices_ai]
df_test_aipastiche = AIpastiche.iloc[test_indices_ai]

# Split NGD
df_train_ngd, df_test_ngd = train_test_split(NGD, test_size=0.2, random_state=42)

# Calcolo offset per AIpastiche in mixed_df
offset = len(NGD)

# Indici originali in mixed_df
train_indices = list(df_train_ngd.index) + [offset + i for i in train_indices_ai]
test_indices = list(df_test_ngd.index) + [offset + i for i in test_indices_ai]

# Funzione per creare dataset con indici coerenti
def build_combined_dataset(human_df, ai_df, human_idx, ai_idx):
    human_part = pd.DataFrame({
        "image_path": human_df["objectid"].values,
        "label": "Human-generated image"
    }, index=human_idx)

    ai_part = pd.DataFrame({
        "image_path": ai_df["generated_image"].values,
        "label": "AI-generated image"
    }, index=ai_idx)

    return pd.concat([human_part, ai_part]).sort_index(kind="merge")

In [ ]:
# Costruzione dei dataset train/test
df_train = build_combined_dataset(df_train_ngd, df_train_aipastiche, df_train_ngd.index, [offset + i for i in train_indices_ai])
df_test = build_combined_dataset(df_test_ngd, df_test_aipastiche, df_test_ngd.index, [offset + i for i in test_indices_ai])

# Verifica
print("Indici df_train:", df_train.index[:10])
print("Indici df_test:", df_test.index[:10])

Indici df_train: Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')
Indici df_test: Index([23, 30, 31, 33, 39, 44, 49, 59, 60, 63], dtype='int64')


In [ ]:
# Human - AI labels
label_to_idx = {"Human-generated image": 0, "AI-generated image": 1}
idx_to_label = {0: "Human-generated image", 1: "AI-generated image"}
labels = ["Human-generated image", "AI-generated image"]

print(f"Train set: {df_train.shape}")
print(f"Test set: {df_test.shape}")

train_image_indices = df_train.index.tolist()
test_image_indices = df_test.index.tolist()

Train set: (1522, 2)
Test set: (384, 2)


In [ ]:
def test_logistic_regression(image_features,n_components=50):
    # filter the image features
    train_image_features = image_features[train_image_indices]
    # val_image_features = image_features[val_image_indices]
    test_image_features = image_features[test_image_indices]

    # convert to float32
    train_image_features = train_image_features.to(dtype=torch.float32)
    test_image_features = test_image_features.to(dtype=torch.float32)

    X_train = train_image_features.cpu().numpy()
    pca = PCA(n_components=n_components)
    pca = pca.fit(X_train)
    X_train = pca.transform(X_train)
    X_test = test_image_features.cpu().numpy()
    X_test = pca.transform(X_test)

    y_train = [label_to_idx[label] for label in df_train['label']]
    y_test = [label_to_idx[label] for label in df_test['label']]

    # 1. Logistic Regression (baseline)
    logreg = LogisticRegression(random_state=42, max_iter=1000, fit_intercept=False, class_weight='balanced')
    logreg.fit(X_train, y_train)
    #print(f"Logistic Regression: downscale = {d}")
    print_test_results(logreg, X_test, y_test)
    print(f"\n")
    return pca,logreg

pca,logreg = test_logistic_regression(image_features,n_components=2)

#AI_centroid = np.mean(image_features_AI.numpy(),axis=0)
#NGA_centroid = np.mean(image_features_NGAD.numpy(),axis=0)

pca_AI = pca.transform(image_features_AI)
pca_NGA = pca.transform(image_features_NGAD)

np.save("pca_AI.npy",pca_AI)
np.save("pca_NGA.npy",pca_NGA)

Accuracy: 0.8932291666666666
Precision: 0.8950767633442266
Recall: 0.8932291666666666
F1: 0.8931255373242385




In [ ]:
ai_points = np.load("pca_AI.npy")
nga_points = np.load("pca_NGA.npy")
ai_image_paths = image_list
nga_image_paths = nga_image_list

In [ ]:
#not used
def encode_image_resized(path, max_size=(64, 64)):
    img = Image.open(path)
    img.thumbnail(max_size)  # resize in-place
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f'<img src="data:image/png;base64,{encoded}" width="100">'

In [ ]:
#not used: attempt to show low resolution image when hoverving
thumbnail_dir = "/content/thumbnails"
os.makedirs(thumbnail_dir, exist_ok=True)

def make_thumbnail(in_path, size=(64, 64)):
    out_path = os.path.join(thumbnail_dir, os.path.splitext(os.path.basename(in_path))[0] + ".jpg")
    if not os.path.exists(out_path):
        img = Image.open(in_path)
        img.thumbnail(size)

        # Convert to RGB, dropping alpha channel (transparency will appear black)
        if img.mode != "RGB":
            img = img.convert("RGB")

        img.save(out_path, format="JPEG")
    return out_path

In [ ]:
#not used
# thumbnail_paths_ai = [make_thumbnail(p) for p in image_list]
# thumbnail_paths_nga = [make_thumbnail(p) for p in nga_image_list]

I wasn't able to get a more interactive version on colab, sorry

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# --- Combine AI and NGA full image paths ---
df_ai = pd.DataFrame({
    "x": ai_points[:, 0],
    "y": ai_points[:, 1],
    "image_path": image_list,
    "label": [f"AI {i}" for i in range(len(ai_points))],
    "source": "AI"
})

df_nga = pd.DataFrame({
    "x": nga_points[:, 0],
    "y": nga_points[:, 1],
    "image_path": nga_image_list,
    "label": [f"NGA {i}" for i in range(len(nga_points))],
    "source": "NGA"
})

df_all = pd.concat([df_ai, df_nga], ignore_index=True)

# --- Plot the PCA scatter plot ---
fig = go.Figure()

for source, color in [("AI", "tomato"), ("NGA", "royalblue")]:
    df = df_all[df_all["source"] == source]
    fig.add_trace(go.Scatter(
        x=df["x"],
        y=df["y"],
        mode="markers",
        marker=dict(size=8, color=color),
        text=df["label"],
        hoverinfo="text",
        name=source
    ))

fig.update_layout(
    title="Type a label (e.g. 'AI 42') to view the image",
    xaxis_title="PCA Component 1",
    yaxis_title="PCA Component 2",
    width=900,
    height=500
)

fig.show()

# --- Text input widget ---
input_box = widgets.Text(
    value='',
    placeholder='Type AI 0 or NGA 17...',
    description='Label:',
    layout=widgets.Layout(width='50%')
)

out = widgets.Output()

def on_submit(change):
    with out:
        out.clear_output()
        label = change['new'].strip()
        match = df_all[df_all["label"] == label]
        if match.empty:
            print(f"No image found for label: {label}")
        else:
            img_path = match.iloc[0]["image_path"]
            img = Image.open(img_path)
            plt.imshow(img)
            plt.axis("off")
            plt.title(label)
            plt.show()

input_box.observe(on_submit, names='value')

display(input_box, out)


Text(value='', description='Label:', layout=Layout(width='50%'), placeholder='Type AI 0 or NGA 17...')

Output()